<a href="https://colab.research.google.com/github/TRPG-masterbot/data/blob/main/Kim/sce_rag_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시나리오 RAG (잠긴 종) — v5

시나리오 카드(`cards.json`)를 검색해서, 지금 플레이어가 볼 수 있는 카드만 가져오는 RAG.

**v5 변경점** — 웹 게임에 붙여 보니 "지금 있는 방에 없는 인물이 나오는" 문제가 있었다. 원인은 검색 필터가 '가 본 방'만 보고 '지금 서 있는 방'은 안 보기 때문이다.
v5에서는 ① state에 현재 방(`room`)을 추가해 **현재 방의 카드만 검색·서술**하게 하고(##6, ##7, ##11), ② 방을 골라 자유 입력으로 확인하는 **테스트 셀**(##14)을 추가했다.
v4와 달라진 곳에는 `# [v5]` 표시가 있다.


| 목차 | 하는 일 | 셀 |
|---|---|---|
| L1 | 카드 읽기, 공개/GM 전용 나누기 | ##1~##2 |
| L2 | 현재 상태에서 검색 가능한 카드만 거르는 규칙 | ##3 |
| L3 | 카드 → 검색용 청크 | ##4 |
| L4 | 임베딩 + Chroma 저장 | ##5 |
| L5 | 상태 필터 검색 (**v5: 현재 방 필터**) | ##6 |
| L6 | 골든셋 평가 (Hit@1, Hit@3, MRR) | ##7~##8 |
| L7 | 자유 입력으로 검색 결과만 확인 (모델 없음) | ##9 |
| L8 | 모델 준비 → 서술 규칙 프롬프트 → 자유 입력으로 모델 답 확인 (**v5: 장소 규칙**) | ##10~##13 |
| L9 | **[v5]** GM 전용 카드 분류표 | ##14 |
| L10 | **[v5]** 현재 방을 골라 자유 입력 테스트 | ##15 |


In [ ]:
# ===== ##0 설치와 import 한 곳에 모으기 (런타임을 바꾸면 이 셀부터 실행) =====
!pip install -q chromadb

import json
import requests
from collections import Counter

import chromadb
from sentence_transformers import SentenceTransformer


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


# L1. 시나리오 카드 정리 (##1~##2)

카드 파일을 읽고, 검색 대상(플레이어에게 보여 줄 수 있는 카드)과 GM 전용 카드를 나눕니다.

비밀이 검색에 섞이지 않게 하는 것이 목적.

In [ ]:
## 1. cards.json 가져오기 + 구조 확인

CARDS_URL = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/Kim/TheDrownedBell/game/data/cards.json"
response = requests.get(CARDS_URL, timeout=60)
response.raise_for_status()
with open("cards.json", "wb") as f:
    f.write(response.content)

with open("cards.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cards = data["cards"]      # 카드 166장
rooms = data["rooms"]      # 방 id -> 한글 이름표 (27개)

print("카드 개수 :", len(cards), "| 방 개수 :", len(rooms))
print("카드 하나의 키 :", list(cards[0].keys()))
for key in ["kind", "owner", "scene_id"]:
    print(key, dict(Counter(card[key] for card in cards)))

# <출력>
# 카드 개수 : 166 | 방 개수 : 27
# kind {'desc': 33, 'npc': 10, 'clue': 55, 'secret': 15, 'check': 30, 'monster': 9, 'ending': 9, 'rumor': 5}
# owner {'all': 133, 'cleric': 11, 'rogue': 8, 'wizard': 9, 'fighter': 5}
# scene_id {1: 34, 2: 16, 3: 37, 4: 28, 5: 26, 6: 25}

카드 개수 : 166 | 방 개수 : 27
카드 하나의 키 : ['id', 'scene_id', 'reveal_after', 'owner', 'kind', 'req_flag', 'text', 'room', 'label', 'player_text', 'gm_only_text', 'gm_note', 'source']
kind {'desc': 33, 'npc': 10, 'clue': 55, 'secret': 15, 'check': 30, 'monster': 9, 'ending': 9, 'rumor': 5}
owner {'all': 133, 'cleric': 11, 'rogue': 8, 'wizard': 9, 'fighter': 5}
scene_id {1: 34, 2: 16, 3: 37, 4: 28, 5: 26, 6: 25}


In [ ]:
## 2. 공개용 / GM 전용 나누기 + 검증
# player_text가 있는 카드만 검색 대상으로 삼는다 (GM 전용 카드는 player_text가 비어 있음)

search_cards = [c for c in cards if c["player_text"]]
print("검색 대상 카드 :", len(search_cards), "| GM 전용 카드 :", len(cards) - len(search_cards))

# 비밀 종류 카드가 섞이지 않았는지 검증
kinds = Counter(c["kind"] for c in search_cards)
for banned in ["secret", "check", "monster"]:
    assert banned not in kinds, banned
print("secret / check / monster 카드가 검색 대상에 없음을 확인")

# 공개 카드인데 GM 문장이 일부 섞인 카드 (이 문장은 청크에 넣지 않는다)
print("GM 문장이 섞인 공개 카드 :", [c["id"] for c in search_cards if c["gm_only_text"]])

# <출력>
# 검색 대상 카드 : 112 | GM 전용 카드 : 54
# GM 문장이 섞인 공개 카드 9장

검색 대상 카드 : 112 | GM 전용 카드 : 54
secret / check / monster 카드가 검색 대상에 없음을 확인
GM 문장이 섞인 공개 카드 : ['scn-s1-npc-halvik', 'scn-s1-clue-well', 'scn-s2-clue-newstone', 'scn-s3-clue-bodies', 'scn-s3-clue-door', 'scn-q4-payoff-lullaby', 'scn-h-fighter-periapt', 'scn-h-rest-infirmary', 'scn-m-npc-innkeeper']


# L2. 검색 규칙 정하기 (##3)

"지금 이 플레이어에게 보여도 되는 카드인가?"를 판단하는 함수.

의미가 비슷해도 아직 공개 전이거나, 가 본 적 없는 방이거나, 다른 직업 전용이면 검색에서 제외.

In [ ]:
## 3. 현재 상황에서 검색 가능한 카드만 거르기

def can_search(card, state): # 필터
    if state["scene"] < card["reveal_after"]:
        return False        # 아직 공개 장면이 아님
    if card["req_flag"] and card["req_flag"] not in state["flags"]:
        return False        # 필요한 플래그가 없음 (req_flag가 빈 글자면 이 검사는 건너뜀)
    if card["room"] and card["room"] not in state["visited"]:
        return False        # 아직 가 보지 않은 방
    if card["owner"] != "all" and card["owner"] != state["cls"]:
        return False        # 다른 직업 전용 카드
    return True


# L3. 시나리오 청크 만들기 (##4)

카드 1장 = 청크 1개

In [ ]:
## 4. 청크 만들기 + 검증

card_by_id = {c["id"]: c for c in cards}      # id로 카드 원본을 바로 찾기 위한 사전

def make_chunk_text(card):
    # "[종류] 방이름" 머리말 + player_text (GM 문장 제외)
    room_name = rooms.get(card["room"], "")   # 엔딩 카드 9장은 room이 비어 있어서 .get 사용
    header = f"[{card['label']}]"
    if room_name:
        header += f" {room_name}"
    return header + "\n" + card["player_text"]

chunks = [{"id": c["id"], "text": make_chunk_text(c)}
          for c in search_cards]

# 검증: 개수 / GM 문장 없음 / id 중복 없음
print("청크 개수 :", len(chunks))
for c, ch in zip(search_cards, chunks):
    if c["gm_only_text"]:
        assert c["gm_only_text"] not in ch["text"], c["id"]
print("GM 문장이 청크에 없음을 확인")
print("id 중복 :", len(chunks) - len({ch["id"] for ch in chunks}))

lengths = sorted(len(ch["text"]) for ch in chunks)
print("청크 길이 min / 중앙 / max :", lengths[0], lengths[len(lengths) // 2], lengths[-1])
print()
print(chunks[0]["text"])

청크 개수 : 112
GM 문장이 청크에 없음을 확인
id 중복 : 0
청크 길이 min / 중앙 / max : 92 153 304

[장면 묘사] 우물 광장
에코 타운. 우물 속 소리가 메아리처럼 올라오던 마을이라 그렇게 불렸다. 채석장이 말라붙은 뒤로 스무 가구만 남은 국경 마을. 마을 한가운데 돌 뚜껑을 덮은 마른 우물이 있고, 그 주위만 풀이 자라지 않는다. 반년 전부터 우물물이 끊겼고, 밤이면 땅 밑에서 종소리가 희미하게 울린다. 종소리는 불규칙하고, 물이 출렁이는 듯 둔탁하다.


# L4. 임베딩과 Chroma 저장 (##5)

청크 글을 벡터(숫자 384개)로 바꿔서 Chroma에 저장.

임베딩 모델: `intfloat/multilingual-e5-small` (가벼운 기본 모델)

In [ ]:
## 5. 임베딩 + Chroma 저장

model = SentenceTransformer("intfloat/multilingual-e5-small") # 가벼운 기본모델

texts = ["passage: " + ch["text"] for ch in chunks]
vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=True) # 모든 청크를 한꺼번에 벡터화
print("벡터 모양 :", vectors.shape)

client = chromadb.Client()      # 메모리에만 저장 (런타임을 끄면 사라짐)
collection = client.get_or_create_collection(
    name="scenario_cards",
    metadata={"hnsw:space": "cosine"},
)
collection.upsert(
    ids=[ch["id"] for ch in chunks],
    documents=[ch["text"] for ch in chunks],
    embeddings=vectors.tolist(),
)
print("저장된 개수 :", collection.count())

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

벡터 모양 : (112, 384)
저장된 개수 : 112


# L5. 시나리오 검색 + 상태 필터 (##6)

Chroma에서 유사도 순으로 전부 가져온 뒤

`can_search`를 통과한 카드만 남기고 앞에서 k개를 돌려줍니다.

In [ ]:
## 6. 상태를 반영한 검색 함수
# 역할: 질문과 비슷한 카드를 찾되, ① 지금 볼 수 있는 카드만(can_search) ② [v5] 지금 서 있는 방의 카드만 남긴다.

ROOM_ONLY = True    # [v5] True: 현재 방(state["room"])의 카드와 방 없는 카드(엔딩)만 검색 / False: v4처럼 가 본 방 전체

def search(question, state, k=3, use_filter=True):
    q_vec = model.encode("query: " + question, normalize_embeddings=True)
    res = collection.query(query_embeddings=[q_vec.tolist()], n_results=len(chunks))
        # 질문을 벡터로 바꿔 카드 전체를 유사한 순서로 가져옴

    room_now = state.get("room")                  # [v5] 지금 서 있는 방 (없으면 None -> 방 제한을 걸지 않음)
    # 비슷한 순서대로 훑으면서, 지금 볼 수 있는 카드만 남겨 앞에서 k개를 돌려준다
    hits = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        card = card_by_id[cid]
        if use_filter and not can_search(card, state):
            continue                              # 지금 볼 수 없는 카드는 건너뜀
        if use_filter and ROOM_ONLY and room_now and card["room"] and card["room"] != room_now:
            continue                              # [v5] 다른 방의 카드는 건너뜀 (방이 없는 엔딩 카드는 통과)
        hits.append((cid, round(1 - dist, 3)))    # (카드 id, 유사도)
    return hits[:k]

# 간단 시험
st = {"scene": 1, "flags": set(), "visited": {"village_square"}, "cls": "fighter"}
for q in ["우물에 돌을 떨어뜨려 본다", "마을 사람에게 말을 건다"]:
    print(q, "->", search(q, st))


우물에 돌을 떨어뜨려 본다 -> [('scn-s1-desc', 0.865), ('scn-s1-clue-missing', 0.851), ('scn-q1-npc-sera', 0.837)]
마을 사람에게 말을 건다 -> [('scn-v8-rumor-bell', 0.855), ('scn-s1-desc', 0.854), ('scn-g-rumors', 0.853)]


# L6. 골든셋과 검색 평가 (##7~##8)

"이 질문에는 이 카드가 정답"이라고 적어 둔 시험 문제집으로 검색을 숫자로 잽니다.
- `golden` : 개발용 14문제
- `golden_mine` : 직접 만든 평가용 4문제

유사도와 순위는 참고용. 최종 판단은 가져온 카드를 직접 읽고함.

In [ ]:
## 7. 골든셋 만들기
# 역할: 시험용 상태(make_state)와 골든셋(질문-정답 카드)을 만든다. [v5] 상태에 현재 방(room)을 추가.

def make_state(scene, cls, room=None, inventory=()):
    # 그 장면까지 열린 방을 모두 가 본 상태로 만든다
    visited = {c["room"] for c in cards if c["reveal_after"] <= scene and c["room"]}
    if room:
        visited.add(room)                 # [v5] 지금 서 있는 방도 가 본 방에 들어 있어야 한다
    return {"scene": scene, "flags": set(), "visited": visited, "cls": cls,
            "room": room,                 # [v5] 지금 서 있는 방 (None이면 방 제한 없음 -> 골든셋 점수는 v4와 같다)
            "inventory": set(inventory)}  # [v5] 소지품 (아직 검색에는 쓰지 않음, 다음 단계용)

golden = [
    {"q": "우물을 덮은 돌판을 자세히 살핀다",          "scene": 1, "cls": "fighter", "answers": ["scn-s1-clue-well"]},
    {"q": "촌장에게 마을에서 있었던 일을 물어본다",    "scene": 1, "cls": "fighter", "answers": ["scn-s1-npc-halvik"]},
    {"q": "눈이 안 보이는 할머니에게 말을 걸어 본다",  "scene": 1, "cls": "fighter", "answers": ["scn-s1-npc-mera"]},
    {"q": "사라진 사람들에 대해 알아본다",             "scene": 1, "cls": "fighter", "answers": ["scn-s1-clue-missing"]},
    {"q": "마을 사람들이 무슨 소문을 말하는지 듣는다", "scene": 1, "cls": "fighter", "answers": ["scn-g-rumors", "scn-v8-rumor-bell"]},
    {"q": "대장장이에게 무기를 사고 싶다",             "scene": 1, "cls": "fighter", "answers": ["scn-m-npc-smith", "scn-m-shop-goods"]},
    {"q": "여관에서 쉬어 가고 싶다",                   "scene": 1, "cls": "fighter", "answers": ["scn-m-npc-innkeeper"]},
    {"q": "촌장 집 벽난로 위에 걸린 물건을 본다",      "scene": 1, "cls": "cleric",  "answers": ["scn-s1-clue-crest", "scn-g-halvik-house"]},
    {"q": "촌장의 손을 유심히 살핀다",                 "scene": 1, "cls": "rogue",   "answers": ["scn-s1-clue-hands"]},
    {"q": "밤에 우물가에서 들리는 소리에 귀를 기울인다", "scene": 1, "cls": "fighter", "answers": ["scn-g-well-night"]},
    {"q": "우물 벽을 타고 내려간다",                   "scene": 2, "cls": "fighter", "answers": ["scn-s2-desc", "scn-g-shaft-stone"]},
    {"q": "중간 턱의 옆 통로를 조사한다",              "scene": 2, "cls": "fighter", "answers": ["scn-s2-clue-newstone"]},
    {"q": "공기가 어디서 흘러오는지 느껴 본다",        "scene": 2, "cls": "wizard",  "answers": ["scn-s2-clue-airflow"]},
    {"q": "갱도 바닥에 숨겨진 것이 있는지 찾아본다",   "scene": 2, "cls": "fighter", "answers": ["scn-h-fighter-periapt"]},
]

# 평가용 (이 문제를 보고 코드를 고치지 않기)
golden_mine = [
    {"q": "최근에 실종한 사람이 있냐고 물어본다",     "scene": 1, "cls": "fighter", "answers": ["scn-s1-clue-missing"]},
    {"q": "할빅의 손을 유심히 본다",                 "scene": 1, "cls": "rogue",   "answers": ["scn-s1-clue-hands"]},
    {"q": "인형을 안 받고 찾아주겠다고 약속한다",     "scene": 1, "cls": "fighter", "answers": ["scn-q1-npc-sera"]},
    {"q": "할빅에게 지금까지 있었던일을 물어본다",   "scene": 2, "cls": "fighter", "answers": ["scn-s1-npc-halvik"]},
]

# 정답 카드가 그 상황에서 검색 가능한지 확인
for g in golden + golden_mine:
    st = make_state(g["scene"], g["cls"])
    for a in g["answers"]:
        assert can_search(card_by_id[a], st), (g["q"], a)
print("골든셋 :", len(golden), "+", len(golden_mine), "문제 / 정답 카드가 모두 그 상황에서 검색 가능함을 확인")

골든셋 : 14 + 4 문제 / 정답 카드가 모두 그 상황에서 검색 가능함을 확인


In [ ]:
## 8. 평가 함수 + 필터 없음/있음 비교

def evaluate(items, use_filter=True):
    ranks = []
    for g in items:
        st = make_state(g["scene"], g["cls"])
        hits = [cid for cid, _ in search(g["q"], st, k=10, use_filter=use_filter)]
                                  # search = can_search 필터
        rank = None
        for i, cid in enumerate(hits):
            if cid in g["answers"]:
                rank = i + 1              # 정답이 처음 나온 등수
                break
        ranks.append(rank)

    n = len(ranks)
    hit1 = sum(r == 1 for r in ranks) / n                          # 1등으로 찾은 비율
    hit3 = sum(r is not None and r <= 3 for r in ranks) / n        # 3등 안에 찾은 비율
    mrr = sum(1 / r for r in ranks if r) / n                       # 등수 역수의 평균
    return ranks, hit1, hit3, mrr

for name, items in [("개발용 14문제", golden), ("직접 만든 4문제", golden_mine)]:
    for use_filter in (False, True):
        ranks, hit1, hit3, mrr = evaluate(items, use_filter)
        print(f"{name} | 필터 {'있음' if use_filter else '없음'} | Hit@1 {hit1:.2f} | Hit@3 {hit3:.2f} | MRR {mrr:.2f}")

# print()
# ranks = evaluate(golden, True)[0]
# for g, r in zip(golden, ranks):
#     print(r, "|", g["q"])

# <이전 실행 결과>
# 개발용 14 : 필터 없음 0.43 / 0.79 / 0.61  ->  필터 있음 0.64 / 0.86 / 0.76
# 직접 만든 4 : 필터 없음 Hit@3 0.75, MRR 0.80  ->  필터 있음 Hit@3 1.00, MRR 0.88

개발용 14문제 | 필터 없음 | Hit@1 0.43 | Hit@3 0.79 | MRR 0.61
개발용 14문제 | 필터 있음 | Hit@1 0.64 | Hit@3 0.86 | MRR 0.76
직접 만든 4문제 | 필터 없음 | Hit@1 0.75 | Hit@3 0.75 | MRR 0.80
직접 만든 4문제 | 필터 있음 | Hit@1 0.75 | Hit@3 1.00 | MRR 0.88


# L7. 자유 입력으로 직접 확인 (##9)

플레이어처럼 하고 싶은 행동을 입력하면, 지금 상황(장면·직업)에서 **볼 수 있는 카드 중 가장 비슷한 카드**를 보여 줍니다.

모델이 글을 만들지 않고, 검색된 카드만 그대로 보여 줍니다. (비어 있는 줄을 입력하면 실행종료)

In [ ]:
## 9. 플레이어처럼 자유 행동 입력해 보기 (빈 줄 입력하면 끝)
# 장면/직업을 바꾸려면 아래 값만 고치세요. 장면 1은 마을, 장면 2는 우물 아래입니다.
SCENE = 1
CLS = "fighter"        # fighter / cleric / rogue / wizard
USE_FILTER = True      # False로 바꾸면 상태 필터 없이 찾는다 (필터 효과 비교용)
K = 3                  # 보여 줄 카드 수
ROOM = None            # [v5] 지금 서 있는 방 id (예: "village_square"). None이면 방 제한 없음

st = make_state(SCENE, CLS, ROOM)
print(f"장면 {SCENE} | 직업 {CLS} | 방 {ROOM} | 필터 {'있음' if USE_FILTER else '없음'}")
while True:
    text = input("\n[자유 행동] > ").strip()
    if not text:
        break
    hits = search(text, st, k=K, use_filter=USE_FILTER)
    print("가져온 카드:")
    for rank, (cid, sim) in enumerate(hits, 1):
        card = card_by_id[cid]
        print(f"  {rank}. {cid} | 유사도 {sim} | 종류 {card['kind']}")
        print("     " + card["player_text"][:80].replace("\n", " ") + "...")


장면 1 | 직업 fighter | 방 None | 필터 있음

[자유 행동] > 


# L8. 모델이 답하게 하기 (##10~##12)

검색으로 찾은 카드(`player_text`)만 모델에게 주고, 모델이 장면을 서술하게 함.


이 노트북에는 게임 엔진이 없어서 소지품·현재 위치는 모릅니다.

지금 방에 없는 사람 거절 같은 코드 규칙도 없고, 검색 필터와 프롬프트 규칙만 씁니다.

룰북 질문은 다루지 않습니다.

In [ ]:
# COLAB-ONLY
# ===== ##10 모델 준비 (Ollama 설치 -> 서버 켜기 -> 모델 받기, 처음엔 몇 분 걸림) =====
import os, re, time, subprocess

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL = "qwen2.5:14b"
NUM_CTX = 8192

# GPU 확인
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU :", gpu or "없음 (런타임을 T4 GPU로 바꾸세요)")

!apt-get install -y -qq zstd
for attempt in range(1, 5):                     # 다운로드가 끊기면 다시 받는다
    print(f"--- Ollama 설치 시도 {attempt}/4 ---")
    result = subprocess.run("curl -fsSL http://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
    if result.returncode == 0 and "Unexpected EOF" not in result.stderr + result.stdout and os.path.exists("/usr/local/bin/ollama"):
        print("설치 완료")
        break
    time.sleep(5)
else:
    raise RuntimeError("Ollama 설치가 계속 실패했어요. 잠시 뒤 이 셀을 다시 실행하세요.")

os.environ["OLLAMA_KEEP_ALIVE"] = "1h"
log = open("ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log, start_new_session=True)   # 서버를 노트북과 따로 켠다
for _ in range(120):
    try:
        if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
            print("서버 준비 완료")
            break
    except Exception:
        pass
    time.sleep(1)

!ollama pull {GEN_MODEL}

def ask_llm(messages, max_new_tokens=300, seed=42, temperature=0.0):
    res = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json={
        "model": GEN_MODEL, "messages": messages, "stream": False,
        "options": {"temperature": temperature,      # 0 = 같은 입력 -> 같은 출력 (재시도할 때는 올린다)
                    "seed": seed, "num_ctx": NUM_CTX, "num_predict": max_new_tokens}})
    res.raise_for_status()
    return res.json()["message"]["content"].strip()

print(ask_llm([{"role": "user", "content": "안녕, 한 문장으로 자기소개 해줘."}]))


GPU : 없음 (런타임을 T4 GPU로 바꾸세요)
Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
--- Ollama 설치 시도 1/4 ---
설치 완료
서버 준비 완료

안녕하세요, 저는 다양한 질문과 작업에 대해 도움을 주는 인공지능 어시스턴트 Qwen입니다.


In [ ]:
# ===== ##11 서술 규칙 프롬프트 + 검사 + 한 턴 함수 =====
# 역할: 검색한 카드만 근거로 모델이 서술하게 하고, 규칙을 어기면 다시 쓰게 한다. [v5] 장소를 현재 방 기준으로.
GM_SYSTEM = ("You are the game master of a Korean single-player D&D adventure. Write every reply in natural Korean (Hangul). "
             "Never use Chinese characters, Japanese, or any other language.")

GM_TEMPLATE = """당신은 1인용 TRPG의 게임 마스터입니다. 아래 규칙을 지키세요.
1. [참고 자료]에 있는 내용만 사용해 지금 장면을 묘사하세요. 자료에 없는 인물, 장소, 사건, 물건을 지어내지 마세요.
2. 플레이어를 '당신'이라고 부르세요. 당신의 행동, 생각, 감정, 대사를 대신 정하지 마세요.
3. 행동은 시도하는 모습과 눈에 보이는 반응까지만 쓰세요. 물건을 받거나 얻는 것, 길이 열리는 것, 누가 쓰러지거나 동의하는 것 같은 결과는 쓰지 마세요. 결과는 게임이 따로 정합니다.
4. 입력에 답할 수 있는 내용이 [참고 자료]에 없으면, 지금은 새로 알 수 있는 것이 없다고 짧게 쓰세요.
5. NPC는 참고 자료에 적힌 성격대로 짧게 말하게 하되, 자료에 없는 사실을 말하게 하지 마세요.
6. 플레이어가 지난 일을 물으면 [최근 기록]에 있는 내용으로 짧게 알려 주세요. 거기 없으면 기억나지 않는다고 하세요.
7. 2~4문장으로 쓰세요. HP, DC 같은 게임 수치와 출처는 쓰지 마세요. 마지막에 "어떻게 하시겠어요?" 같은 질문은 쓰지 마세요.
8. [장소]에 없는 인물이나 장소는 등장시키지 마세요. 지금 이 방에서 보이는 것만 쓰세요.

[장소] {room}

[참고 자료]
{docs}

[최근 기록]
{history}

[플레이어] {action}
[GM]"""

# 서술을 검사하는 규칙 (걸리면 문제를 알려 주고 다시 쓰게 한다)
BROKEN = re.compile(r"[^가-힣ㄱ-ㆎ\x00-\x7f\s…·“”‘’—–]")        # 한글·영문·기호 말고 다른 글자 (한자 등)
LEAK = re.compile(r"\b(HP|DC|CR|SRD)\b|p\.\s*\d+|scn-")                          # 게임 수치·출처 노출
OUTCOME = re.compile(r"(건네|건넨|넘겨|내어\s*준|내준|받았|받는다|획득|쥐여|꺼내\s*준|얻었|얻는다|열렸|쓰러졌|쓰러진다)")   # 결과를 확정하는 말
NO_NEWS = "지금은 이 행동을 할 수 없을 거 같다. 다른 행동을 해 보자."

def narration_problems(body):
    p = []
    if BROKEN.search(body): p.append("깨진 글자")
    if LEAK.search(body): p.append("게임 수치·출처 노출")
    if len(body) < 20: p.append("너무 짧음")
    if re.search(r"플레이어|주인공", body): p.append("호칭")
    if OUTCOME.search(body): p.append("행동의 결과를 확정함")
    return p

def clean_body(body):
    # 끝까지 문제가 남았을 때: 문제 있는 문장을 빼고, 남는 게 없으면 한 줄 안내
    body = re.sub(r"플레이어|주인공", "당신", body)
    sents = re.split(r"(?<=[.!?…])\s+|\n+", body)
    kept = " ".join(s for s in sents if s and not (BROKEN.search(s) or LEAK.search(s) or OUTCOME.search(s))).strip()
    return kept if len(kept) >= 20 else NO_NEWS

def recent_records(history, n=6):
    return "\n".join(f"플레이어: {a} / GM: {b}" for a, b in history[-n:]) or "(없음)"

def narrate_scene(text, state, history, k=2, tries=3):
    hits = search(text, state, k=k)                                    # 상태 필터가 적용된 시나리오 검색
    if not hits:                                                       # 볼 수 있는 카드가 없으면 모델을 부르지 않는다
        return NO_NEWS, hits, {"tries": 0, "problems": []}
    docs = [card_by_id[cid]["player_text"] for cid, _ in hits]          # 플레이어용 글만 모델에게 준다
    room_id = state.get("room")                                        # [v5] 지금 서 있는 방
    if room_id:
        room = rooms.get(room_id, room_id)                             # [v5] 실제 현재 방의 이름 (검색된 카드 기준이 아님)
    else:                                                              # 방을 모르면 v4처럼 가장 위 카드의 방
        rooms_here = [rooms.get(card_by_id[cid]["room"], "") for cid, _ in hits]
        room = next((r for r in rooms_here if r), "(알 수 없음)")
    prompt = GM_TEMPLATE.format(room=room, docs="\n\n".join(f"- {d}" for d in docs) or "(없음)",
                                history=recent_records(history), action=text)
    messages = [{"role": "system", "content": GM_SYSTEM}, {"role": "user", "content": prompt}]
    body, probs = "", []
    for t in range(tries):
        body = ask_llm(messages, max_new_tokens=250, seed=42 + t, temperature=0.7 if t else 0.3)
        probs = narration_problems(body)
        if not probs:
            break
        messages = messages[:2] + [{"role": "assistant", "content": body},
                                   {"role": "user", "content": "다음 문제가 있어. 고쳐서 다시 써 줘: " + "; ".join(probs)}]
    if probs:
        body = clean_body(body)
    body = re.split(r"\s*(?:(?:어떻게|무엇을|뭘)\s*(?:계속|대응)?\s*하시겠|\[게임 진행)", body)[0].rstrip()   # 끝의 되묻기와 군더더기 제거
    return body, hits, {"tries": t + 1, "problems": probs}


In [ ]:
## 12. 플레이어처럼 자유 행동 입력해 보기 - 모델 서술까지 (빈 줄 입력하면 끝)
# 장면/직업을 바꾸려면 아래 값만 고치세요. 장면 1은 마을, 장면 2는 우물 아래입니다.
SCENE = 3
CLS = "fighter"        # fighter / cleric / rogue / wizard
SHOW_CARDS = True      # 모델에게 준 카드도 같이 보여 준다 (False면 서술만)
ROOM = None            # [v5] 지금 서 있는 방 id (예: "village_square"). None이면 방 제한 없음

st = make_state(SCENE, CLS, ROOM)
free_history = []      # (입력, 서술) 기록. 다음 입력의 '최근 기록'으로 쓰인다
print(f"장면 {SCENE} | 직업 {CLS} | 방 {ROOM} | 모델 {GEN_MODEL}")
while True:
    text = input("\n[자유 행동] > ").strip()
    if not text:
        break
    t0 = time.time()
    body, hits, info = narrate_scene(text, st, free_history)
    if SHOW_CARDS:
        print("가져온 카드:")
        for cid, sim in hits:
            print(f"  - {cid} ({sim}) {card_by_id[cid]['player_text'][:50]}...")
    print("\nGM:", body)
    print(f"({time.time() - t0:.1f}초 | 시도 {info['tries']}번 | 남은 문제 {info['problems'] or '없음'})")
    free_history.append((text[:60], body[:120]))

# 장면 1 | 직업 fighter | 모델 qwen2.5:14b

# <출력 결과>
# [자유 행동] > 촌장 할빅을 공격한다
# 가져온 카드:
#   - scn-s1-npc-halvik (0.906) 할빅 — 에코 타운의 촌장. 60대 남성, 전직 채석공. 어깨가 넓고 목소리가 크다. 일행...
#   - scn-g-halvik-house (0.881) 촌장 집. 채석장이 한창일 때 지은 큰 집이라 방이 넓고 허름하다. 벽난로 위에 낡은 석판...

# GM: 할빅이 놀란 표정으로 당신을 노려보고는 뒤로 물러섭니다. 그의 몸짓에서 분노와 경계가 느껴집니다. "당신이 뭔가 잘못 알아들었나 봅니다. 제가 해달라는 일은 우물을 고치는 거예요." 할빅은 다시 말을 이어가려 하지만, 당신의 행동으로 인해 말을 멈춥니다.
# (6.7초 | 시도 1번 | 남은 문제 없음)

# [자유 행동] > 세라에게 딸과 남편이 언제 실종됐는지 물어본다
# 가져온 카드:
#   - scn-q1-npc-sera (0.874) 세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다...
#   - scn-s1-clue-missing (0.848) 실종자 셋. 모두 지난 두 달 사이, 모두 밤에, 모두 우물 근처에서 마지막으로 목격됐다....

# GM: 세라가 눈물을 머금고 말합니다. "토브와 리나는 두 달 전, 바로 이 우물 근처에서 마지막으로 보였어요. 그날 밤에도 우물에서 이상한 종소리가 들렸다고 기억해요. 가렌은 그보다 한 달 전에 사라졌죠." 세라의 목소리가 약간 떨립니다.
# (5.8초 | 시도 1번 | 남은 문제 없음)

# [자유 행동] > 세라의 인형말고 다른 단서는 없는지 물어본다
# 가져온 카드:
#   - scn-q1-npc-sera (0.842) 세라 — 실종자 토브의 아내이자 리나의 어머니. 30대 후반, 우물가에서 매일 밤을 새운다...
#   - scn-g-rumors (0.836) 마을 사람들의 소문. 대장간 노인은 "종소리가 날마다 가까워진다"고 하고, 여관 주인은 밤...

# GM: 세라가 고개를 저으며 말합니다. "이 인형 말고는 다른 단서가 없어요. 그날 밤 이후로 아무런 흔적도 없었어요. 하지만 이 인형이 리나가 찾을 수 있는 유일한 단서일 거예요." 세라의 표정은 절망과 희망 사이에서 갈등하고 있습니다.
# (5.1초 | 시도 1번 | 남은 문제 없음)

# [자유 행동] > 우물가 아래로 인형을 떨어트려본다
# 가져온 카드:
#   - scn-s1-clue-well (0.866) 마른 우물. 돌 뚜껑은 최근에 올렸다 내린 자국이 있다. 이끼가 한쪽만 벗겨져 있다. 뚜껑...
#   - scn-q4-npc-mera-song (0.859) 메라는 물어보지 않아도 우물 곁에 앉아 입속으로 노래를 흥얼거린다. 일행이 곁에 앉아 끝까...

# GM: 당신이 인형을 우물 안으로 던져 보지만, 인형은 어둠 속으로 사라지고 곧장 물에 닿는 소리가 들립니다. 메라는 당신의 행동을 보고 고개를 돌려 잠시 멍하니 바라봅니다.
# (4.9초 | 시도 1번 | 남은 문제 없음)

# [자유 행동] > 메라를 데리고 우물 안으로 뛰어든다
# 가져온 카드:
#   - scn-q4-npc-mera-song (0.886) 메라는 물어보지 않아도 우물 곁에 앉아 입속으로 노래를 흥얼거린다. 일행이 곁에 앉아 끝까...
#   - scn-s1-npc-mera (0.883) 메라 — 우물지기였던 노파. 80대, 눈이 거의 보이지 않는다. 마을에서 유일하게 옛 수도...

# GM: 메라는 당신의 갑작스러운 행동에 깜짝 놀라 눈을 크게 뜨며 말합니다. "어머나, 그만하세요! 여기서 뭐하는 거예요? 우물은 위험해요." 메라는 계속해서 당신을 붙잡으려 하지만, 당신이 이미 우물 가장자리를 넘어서 떨어지는 중입니다. 메라는 당신이 떨어지는 것을 바라보며 크게 소리를 지르며 뒤로 물러섭니다.
# (7.3초 | 시도 1번 | 남은 문제 없음)



장면 3 | 직업 fighter | 방 None | 모델 qwen2.5:14b

[자유 행동] > 


In [ ]:
# ===== ##13 장면 3~6 자유행동 20개 실험 =====
# (장면, 현재 방, 직업, 켜진 플래그, 입력)
CASES = [
    (3, "cloister_entry",      "rogue",   [],                   "물속을 더듬어 반짝이는 걸 찾는다"),
    (3, "cloister_mid",        "fighter", [],                   "회랑 한가운데 물속으로 앞장서서 걸어간다"),
    (3, "cloister_bodies",     "rogue",   [],                   "물속에서 연장을 찾아본다"),
    (3, "scriptorium_door",    "fighter", [],                   "필사실 문을 어깨로 밀어 본다"),
    (3, "infirmary_inner",     "fighter", ["opened_infirmary"], "선반의 병들을 살펴본다"),
    (4, "scriptorium_hall",    "rogue",   [],                   "책상 쪽으로 바닥을 살피며 조심히 걷는다"),
    (4, "scriptorium_desks",   "fighter", [],                   "책상 서랍을 뒤진다"),
    (4, "armor_corner",        "fighter", [],                   "갑옷 발치의 기록물을 집어 든다"),
    (4, "scriptorium_shelves", "wizard",  [],                   "서가의 책들을 훑어 본다"),
    (4, "stairs_down",         "fighter", [],                   "계단에서 들리는 종소리를 세어 본다"),
    (5, "bell_chamber",        "fighter", [],                   "횃불을 높이 들어 물 위를 비춘다"),
    (5, "sluice_works",        "fighter", [],                   "기어 하나를 돌려 본다"),
    (5, "bell_underside",      "cleric",  [],                   "종 아래 물속을 더듬어 본다"),
    (5, "bell_chamber",        "fighter", ["got_rina_doll"],    "리나의 인형을 그림자 앞에 내민다"),
    (5, "bell_tongue",         "rogue",   [],                   "종의 혀 아래 둥지에서 번쩍이는 걸 꺼낸다"),
    (6, "belfry_stair",        "fighter", [],                   "쇠사슬 계단을 한 칸씩 오른다"),
    (6, "belfry_platform",     "fighter", [],                   "종채를 잡고 종을 친다"),
    (6, "belfry_platform",     "fighter", ["found_last_pages"], "할빅에게 칼릭의 이름을 부르라고 한다"),
    (6, "belfry_stair",        "fighter", [],                   "수문을 복구해서 물을 빼겠다고 선언한다"),
    (6, "belfry_stair",        "fighter", [],                   "포기하고 지상으로 올라간다"),
]

for n, (scene, room, cls, flags, text) in enumerate(CASES, 1):
    state = {"scene": scene, "flags": set(flags), "visited": {room}, "cls": cls,
             "room": room, "inventory": set()}      # [v5] 현재 방(room)과 소지품 추가
    body, hits, info = narrate_scene(text, state, [])
    print(f"\n#{n} 장면 {scene} | {room} | {cls}")
    print("입력:", text)
    print("카드:", ", ".join(f"{cid}({sim})" for cid, sim in hits))
    print("검사:", info)
    print("GM:", body)

# L9. 현재 방을 골라 자유 입력 테스트 (##15) — v5 추가

서 있는 방(`ROOM`)을 정해 놓고 자유 행동을 입력하면, **현재 방 제한이 없을 때와 있을 때의 검색 결과를 나란히** 보여 주고 모델 서술까지 확인합니다.
서술까지 보려면 ##10(모델 준비)과 ##11이 먼저 실행되어 있어야 합니다. (빈 줄을 입력하면 끝)


In [ ]:
## 14. 현재 방 기준 자유 행동 테스트 (빈 줄 입력하면 끝)
# 역할: 방을 정해 두고 입력하면 ① 현재 방 제한 없음/있음 검색 결과를 비교하고 ② 모델 서술까지 확인한다.
SCENE = 3
CLS = "fighter"             # fighter / cleric / rogue / wizard
ROOM = "village_square"     # 지금 서 있는 방 id (아래에 출력되는 목록에서 고른다)
K = 2                       # 보여 줄 카드 수
NARRATE = True              # True: 모델 서술까지 (##10~##11 필요) / False: 검색 결과만
NARRATE_ROOM_ONLY = True    # 서술에 쓸 카드에 현재 방 제한을 걸지 (False로 두면 v4처럼 가 본 방 전체)

open_rooms = {c["room"] for c in cards if c["reveal_after"] <= SCENE and c["room"]}
print("이 장면에서 고를 수 있는 방:", {r: rooms[r] for r in sorted(open_rooms)})
assert ROOM in open_rooms, f"ROOM '{ROOM}'은 장면 {SCENE}에서 열린 방이 아닙니다"

def show_cards(hits):
    return " | ".join(f"{cid}({sim}, {rooms.get(card_by_id[cid]['room'], '방 없음')})" for cid, sim in hits) or "(없음)"

st = make_state(SCENE, CLS, ROOM)
history = []                # (입력, 서술) 기록
print(f"\n장면 {SCENE} | 직업 {CLS} | 현재 방 {rooms[ROOM]} ({ROOM})")
while True:
    text = input("\n[자유 행동] > ").strip()
    if not text:
        break
    ROOM_ONLY = False
    print("현재 방 제한 없음:", show_cards(search(text, st, k=K)))     # v4 방식
    ROOM_ONLY = True
    print("현재 방 제한 있음:", show_cards(search(text, st, k=K)))     # v5 방식
    if NARRATE:
        ROOM_ONLY = NARRATE_ROOM_ONLY
        t0 = time.time()
        body, hits, info = narrate_scene(text, st, history, k=K)
        print("\nGM:", body)
        print(f"({time.time() - t0:.1f}초 | 시도 {info['tries']}번 | 남은 문제 {info['problems'] or '없음'})")
        history.append((text[:60], body[:120]))
ROOM_ONLY = True            # 다른 셀을 위해 기본값으로 되돌린다
